# 00 — Environment Setup

> **ICAN CA Training — Generative AI & RAG (6 hours).** This notebook is part of a 14-notebook curriculum. All data is synthetic. Confidential client data must not be used with public APIs without engagement-letter authority. AI output must always be verified by a qualified professional.


## Learning objectives

By the end of this notebook you can:
1. Confirm Python and required libraries are installed.
2. Load your API keys from `.env` and see which provider is active.
3. Confirm the synthetic dataset has been generated.
4. Make one successful call to the configured LLM (or see the mock fallback).


In [1]:
# --- Bootstrap (don't edit) ---
# Adds the project root to sys.path so we can do `from src.xxx import yyy`.
import sys, os
from pathlib import Path
ROOT = Path.cwd()
# Walk up until we find the project root (folder that contains src/)
for _ in range(4):
    if (ROOT / 'src').exists() and (ROOT / 'requirements.txt').exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Project root:', ROOT)


Project root: /Users/aayush/Documents/Kings/ICAN/ICANN/rag


## Step 1 — Check Python version

We target Python 3.10+.

In [2]:
import sys, platform
print('Python:', sys.version.split()[0], 'on', platform.system())
assert sys.version_info >= (3, 9), 'Please use Python 3.9+'

Python: 3.14.5 on Darwin


## Step 2 — Check critical libraries are installed

In [4]:
missing = []
for pkg in ['pandas', 'openpyxl', 'dotenv', 'pypdf', 'numpy', 'tqdm']:
    try:
        __import__(pkg)
    except Exception as e:
        missing.append((pkg, str(e)))
if missing:
    print('Missing packages:', missing)
    print('Fix: open a terminal and run  pip install -r requirements.txt')
else:
    print('Core packages: ok')

Core packages: ok


## Step 3 — Load environment variables and report status

Copy `.env.example` to `.env` and fill in *at least one* provider key. The `llm_status()` helper tells you which provider will be used.

In [5]:
from src.llm_client import llm_status, ask_llm
status = llm_status()
for k, v in status.items():
    print(f'  {k:20s}: {v}')
if status['is_mock']:
    print('\n[note] No real provider key detected. ask_llm() will return MOCK answers.')

  provider            : openai
  model               : gpt-4o-mini
  is_mock             : False
  openai_key_set      : True
  anthropic_key_set   : True
  google_key_set      : True


## Step 4 — Confirm the dataset is on disk

If anything is missing, run `python src/data_generation.py` from the project root.

In [6]:
from pathlib import Path
for d in ['data/generated/pdf', 'data/generated/xlsx', 'data/generated/csv']:
    files = sorted(Path(d).glob('*'))
    print(f'  {d}  →  {len(files)} files')
    for f in files[:3]:
        print('     -', f.name)
    if len(files) > 3:
        print(f'     ... and {len(files)-3} more')

  data/generated/pdf  →  10 files
     - 01_annual_report_extract.pdf
     - 02_audit_planning_memo.pdf
     - 03_internal_control_policy.pdf
     ... and 7 more
  data/generated/xlsx  →  10 files
     - 01_trial_balance.xlsx
     - 02_general_ledger_sample.xlsx
     - 03_fixed_asset_register.xlsx
     ... and 7 more
  data/generated/csv  →  10 files
     - 01_sales_transactions.csv
     - 02_purchase_transactions.csv
     - 03_journal_entries.csv
     ... and 7 more


## Step 5 — One smoke-test LLM call

A single short call to confirm everything is wired.

In [8]:
answer = ask_llm(
    'In one short sentence, explain what materiality means in an audit.',
    system='You are an experienced Nepali Chartered Accountant.'
)
print(answer)

Materiality in an audit refers to the significance of an amount, transaction, or discrepancy that could influence the decision-making of users of the financial statements.


## Expected output

* If a real key is set: a one-sentence professional definition of materiality.
* If no key is set: a `[MOCK LLM]` message echoing your prompt.


## Common errors

| Error | Likely cause | Fix |
|---|---|---|
| `ModuleNotFoundError: dotenv` | Forgot `pip install -r requirements.txt` | Activate the venv and reinstall |
| `Project root: ...` looks wrong | You opened the notebook from a strange folder | Run JupyterLab from the `rag/` folder |
| `AuthenticationError` | Bad API key | Re-open `.env`, check for stray quotes/spaces |
| Empty `data/generated/*` | Forgot to generate data | `python src/data_generation.py` |


## ⚠️ Professional caution

The synthetic dataset is safe to use. The moment you switch to **real client data** (later in Notebook 08), check your engagement-letter terms and prefer the local embedding model (`EMBEDDING_PROVIDER=local`).